# Goal

- Define optional list of columns that can build the AG
- Compare between their statistics
- Choose the most relevant one

For example:
- Option 1: ["brand", "color"]
- Option 2: ["brand", "color", "width"]
- Option 3: ["brand", "color", "weight"]

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from os import path, mkdir

# User Preferences

## Define Thresholds
You can define 2 different thresholds for deciding which combinations to filter automatically
1. min_products_in_group - Minimum number of products in a category
2. max_products_in_group - Maximum number of products in a category


* hist_xlimit = Maximum value to show in the histogram X axis
* hist_xticks_jumps = Intervals of the X ticks in the histogram X axis

In [ ]:
min_products_in_group = 1
max_products_in_group = 100000

hist_xlimit = 100
hist_xticks_jumps = 20

## Columns Combinations
Define here all the combinations you want to test

In [ ]:
combinations = [
    ['brands', 'colors', 'styles'],
    ['brands', 'colors']
]

# Load Data
You need to create a CSV file named as "catalog.csv" and put it in the same directory of this script

In [ ]:
df = pd.read_csv(
    'catalog.csv',
    dtype={
        'id': 'str',
        'name': 'str',
        'description': 'str',
        'product_id': 'str',
        'product_name': 'str',
        'product_description': 'str',
        'categories': 'str',
        'brands': 'str',
        'categories': 'str',
        'colors': 'str',
        'markets': 'str',
        'seasons': 'str',
        'styles': 'str',
        'size': 'str',
        'department_name': 'str',
        'department_id': 'str'
        }
)

print(f'Original file shape: {df.shape}')
df.drop_duplicates(inplace=True)
print(f'Shape after removing duplicated rows: {df.shape}')

display(df.head(2))
df.describe()

In [ ]:
df.dtypes

## Create Relevant Folders
Create the output folder ("output") if not exists

In [ ]:
output_dir = 'output'
if not path.isdir(output_dir):
    mkdir(output_dir)

# Prepare Data

## Fill Missing Values

In [ ]:
if 'avoid_replenishment' in df.columns:
    print('Fill avoid replenishment with 0')
    df['avoid_replenishment'].fillna(0, inplace=True)

In [ ]:
# Fill missing values of price and cost with 0
print(f'Missing price and cost values:\n{df[["price", "cost"]].isna().sum()}')
df['price'].fillna(0, inplace=True)
df['cost'].fillna(0, inplace=True)
print('Missing price & cost values filled with 0')

In [ ]:
# Fill other string columns with empty string
str_cols_ = {'product_id', 'product_name', 'categories', 'brands', 'categories', 'colors', 'markets', 'seasons', 'styles', 'size', 'department_name', 'department_id'}
df_columns = set(df.columns)
relevant_cols = list(str_cols_.intersection(df_columns))
df_missing_values = df[relevant_cols].isna().sum()[df[relevant_cols].isna().sum() > 0]
print(f'Missing values: {df_missing_values}')
cols_to_fill = list(df_missing_values.index)
df[cols_to_fill] = df[cols_to_fill].fillna(value='empty_value')
print('All missing values were filled with "empty_value" string')

## Filter Avoid Replenishment SKUs

In [ ]:
if 'avoid_replenishment' in df.columns:
    filter_ar = input('Do you want to filter Avoided SKUs? (yes / no)').lower() == 'yes'
    print(f'Filtering Avoided SKUs: {filter_ar}')
else:
    filter_ar = False
    print('"avoid_replenishment" column is not in the catalog file - filter is not relevant')

In [ ]:
if filter_ar:
    print(f'# SKUs before the filter: {len(df)}')
    mask = df['avoid_replenishment'] != True
    df = df.loc[mask]
    print(f'Avoided SKUs were filtered out from the catalog.\n# SKUs after the filter: {len(df)}')

## Add Price and Cost groups

Enter the desired number of price and cost groups

In [ ]:
while True:
    try:
        num_price_groups = int(input('Enter number of price groups'))
        num_cost_groups = int(input('Enter number of cost groups'))
        break
    except ValueError:
        print('Please enter a valid number')

In [ ]:
df['price_group'], bins_ = pd.cut(
    x=df['price'],
    bins=num_price_groups,
    retbins=True
)
df['price_group'] = df['price_group'].astype(str)

print('Price groups distribution:')
df.groupby('price_group')['price'].agg(['min', 'max', 'count'])

In [ ]:
df['cost_group'], bins_ = pd.cut(
    x=df['cost'],
    bins=num_cost_groups,
    retbins=True
)
df['cost_group'] = df['cost_group'].astype(str)

print('Cost groups distribution:')
df.groupby('cost_group')['cost'].agg(['min', 'max', 'count'])

In [ ]:
df.head(3)

# Helper Functions
**** You should not change these functions

In [ ]:
def build_category(df, cat_cols):
    df['ag'] = ''
    for c in cat_cols:
        df['ag'] += df[c]

    return df

In [ ]:
def calc_dist(df, comb_str):
    dist_ = df.groupby('ag')['product_id'].size()
    dist_desc = dist_.describe()
    n_groups = dist_.shape[0]
    n_products = dist_.sum()

    min_products_in_group = min(dist_)
    max_products_in_group = max(dist_)

    df_stats = pd.DataFrame({
        'cols': [comb_str],
        '# Products': [n_products],
        '# AGs': [n_groups],
        'min_products_in_ag': [min_products_in_group],
        'max_products_in_ag': [max_products_in_group],
        'Average Products in AG': [int(dist_desc['mean'])],
        'Median Products in AG': [dist_desc['50%']]
    })

    return df_stats, dist_

In [ ]:
def filter_comb(df_stats):
    min_, max_ = df_stats[['min_products_in_ag', 'max_products_in_ag']].values[0]
    if (min_ >= min_products_in_group) & (max_ <= max_products_in_group):
        return True

    return False

In [ ]:
def plot_dist(dist_, df_stats, comb_str):
    # Create a histogram plot using matplotlib
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(dist_, bins=min(df_stats['# AGs'].values[0], 50), range=(0, hist_xlimit))
    ax.set_xlabel('# of products in AG')
    ax.set_ylabel('# of AGs')
    title = f'Distribution of Products per Combination - {comb_str}'
    ax.set_title(title)  # Set the title based on the provided parameter
    plt.xticks(np.arange(min(dist_), hist_xlimit, hist_xticks_jumps), rotation='vertical')  # Rotate x-axis labels for better readability

In [ ]:
def plot_ags(dist_, df_stats, comb_str):
    # Create a histogram plot using matplotlib
    fig, ax = plt.subplots(figsize=(10, 6))
    dist_.sort_values(ascending=False)[:hist_xlimit].plot(kind='bar', ax=ax)
    ax.set_xlabel('AG')
    ax.set_ylabel('# of Products')
    title = f'Products per AG - {comb_str}'
    ax.set_title(title)  # Set the title based on the provided parameter

In [ ]:
def save_comb(df, comb_str):
    """ Extract combination output as a CSV file """
    csv_path = path.join(output_dir, f'{comb_str}.csv')
    if not path.isfile(csv_path):
        df.to_csv(csv_path, index=False)
    print(f"CSV file created: {csv_path}")

In [ ]:
def check_combinations(df, combinations):
    df_stats_list = []

    for comb in combinations:
        comb_str = "_".join(comb)
        df_tmp = build_category(df=df, cat_cols=comb)
        df_stats, dist_ = calc_dist(df=df_tmp, comb_str=comb_str)
        if filter_comb(df_stats):
            print(f'Combination {comb_str} is relevant')
            plot_dist(dist_=dist_, df_stats=df_stats, comb_str=comb_str)
            plot_ags(dist_=dist_, df_stats=df_stats, comb_str=comb_str)
            save_comb(df=df_tmp, comb_str=comb_str)
        else:
            print(f'Combination {comb_str} is not relevant')

        df_stats_list.append(df_stats)

    df_stats_final = pd.concat(df_stats_list)
    return df_stats_final

# Run

In [ ]:
df_stats_final = check_combinations(
    df=df,
    combinations=combinations
)
display(df_stats_final)